# COLUMNS TRANSFORMER. PIPELINE

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

from sklearn.compose import ColumnTransformer
#from sklearn.compose import make_column_selector
from sklearn.pipeline import Pipeline


import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import warnings

warnings.filterwarnings('ignore')

In [31]:
def categoricas_unicos(dataframe, var_cat, vc_out=True):
    """
    Muestra valores unicos y conteo de las variables categóricas.
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables categóricas (cadena)
    - vc_out: booleano para mostrar o no (True/False) el resultado de "value_counts()"
        Por defecto lo muestra

    -------------------------------
    SALIDA:
    No devuelve valor. Saca por pantalla la información

    """

    for discreta in var_cat:

        print(f'Variable {discreta.upper()}:')
        print('Valores unicos: ')
        print(dataframe[discreta].unique(), end='\n'*2)

        if vc_out:
            print(dataframe[discreta].value_counts(), end='\n'*2)





In [32]:
def graficas_var_categorica(dataframe, var_cat):
    """
    Realiza los diagramas de barras de las variables categóricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """


    colores = sns.color_palette("husl", len(var_cat))

    # creacion matriz de graficas
    fig, axes = plt.subplots(len(var_cat), 1, \
                             figsize=(10, 4*len(var_cat)),\
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

    ax = axes.ravel()

    # dibujamos las graficas
    for idx,variable in enumerate(var_cat):

        sns.countplot(dataframe[variable], ax=ax[idx],palette=colores)

        ax[idx].set_title(f'DIAGRAMA DE BARRAS {variable}')
        ax[idx].set_xlabel(f'Valores únicos (categorias) de {variable}')
        ax[idx].set_ylabel("Frequencia")


In [33]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.distplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     hist_kws={'alpha': 0.15})

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')



## CASO MAS SENCILLO

In [34]:
df1 = pd.read_csv('https://raw.githubusercontent.com/SonikoKatsura/CURSO_PARO_IA/refs/heads/main/Rober_IA/Modulo3/datos/adult1.csv')

df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24967 entries, 0 to 24966
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             24967 non-null  int64  
 1   workclass       24967 non-null  object 
 2   marital_status  24967 non-null  object 
 3   sex             24967 non-null  object 
 4   hours_per_week  24967 non-null  int64  
 5   capital         24967 non-null  float64
 6   income          24967 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 1.3+ MB


In [35]:
df1.isnull().sum()

,0
age,0
workclass,0
marital_status,0
sex,0
hours_per_week,0
capital,0
income,0


In [36]:
target = 'income'

X = df1.drop(columns=target)

y = df1[target]

X.shape, y.shape

((24967, 6), (24967,))

In [37]:
atrib_num = X.select_dtypes(exclude='object').columns.to_list()

atrib_num

['age', 'hours_per_week', 'capital']

In [38]:
atrib_cat = X.select_dtypes(include='object').columns.to_list()

atrib_cat

['workclass', 'marital_status', 'sex']

**El target lo podemos tratar ya:**

In [39]:
label_enc = LabelEncoder()

y_enc = label_enc.fit_transform(y)


In [40]:
y_enc

array([0, 0, 0, ..., 1, 0, 1])

**En cuanto a los atributos el tratamiento a seguir es convertir a numérico las categóricas (utilizamos como ejemplo OHE) y escalar las numéricas (Standar Scaler). Veamos como lo hacemos con "ColumnnTransformer":**

In [41]:
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), atrib_num),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), atrib_cat)],
    remainder='passthrough')

In [42]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

((19973, 6), (4994, 6), (19973,), (4994,))

Respetamos la regla de entrenar con TRAIN:

In [43]:
Xtrain_prep = preprocessor.fit_transform(Xtrain)
Xtest_prep  = preprocessor.transform(Xtest)

Xtrain_prep.shape, Xtest_prep.shape

((19973, 14), (4994, 14))

Para recuperarlo como DF, hay que tener en cuenta que en el "Columns Transformer" primero tratamos las numéricas, y despues las categóricas. Al utilizar para éstas últimas OHE, nos salen más columnas de las originales y recuperamos sus nombres del método correspondiente del transformador:

In [44]:
encoded_cat = preprocessor.named_transformers_['onehot'].get_feature_names_out(atrib_cat)

nombre_columnas = np.concatenate([atrib_num, encoded_cat])

nombre_columnas


array(['age', 'hours_per_week', 'capital', 'workclass_Private',
       'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc',
       'workclass_goverment', 'marital_status_Divorced',
       'marital_status_Married', 'marital_status_Never-married',
       'marital_status_Separated', 'marital_status_Widowed', 'sex_Female',
       'sex_Male'], dtype=object)

In [45]:

df_train_prep = pd.DataFrame(Xtrain_prep, columns=nombre_columnas)

df_train_prep.sample(n=10)


,age,hours_per_week,capital,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_goverment,marital_status_Divorced,marital_status_Married,marital_status_Never-married,marital_status_Separated,marital_status_Widowed,sex_Female,sex_Male
6143,0.332783,-0.841166,-0.612445,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
13009,-1.075764,-2.354252,-1.485402,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3312,0.068680,-0.153400,-1.769232,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5886,0.420817,1.222132,0.354101,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
19295,2.269535,-0.153400,-0.739537,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
8975,-0.019354,-0.153400,-1.133941,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
8491,0.684919,1.222132,-0.216945,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
6005,1.037056,-0.153400,0.082952,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
9767,1.565261,-2.216698,-0.233708,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
12561,0.596885,-0.153400,-0.881832,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [46]:
df_train_prep.describe()

,age,hours_per_week,capital,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_goverment,marital_status_Divorced,marital_status_Married,marital_status_Never-married,marital_status_Separated,marital_status_Widowed,sex_Female,sex_Male
count,1.997300e+04,1.997300e+04,1.997300e+04,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000,19973.000000
mean,-3.984418e-17,1.209556e-17,-2.561412e-17,0.708006,0.041806,0.095629,0.154559,0.165123,0.563160,0.203274,0.035147,0.033295,0.298002,0.701998
std,1.000025e+00,1.000025e+00,1.000025e+00,0.454691,0.200152,0.294089,0.361492,0.371301,0.496007,0.402445,0.184157,0.179410,0.457392,0.457392
min,-1.427901e+00,-2.904464e+00,-3.448372e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-8.116617e-01,-1.534001e-01,-6.456520e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.073882e-01,-1.534001e-01,-1.504982e-01,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,6.849194e-01,9.470256e-01,3.671801e-01,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000
max,4.206287e+00,1.222132e+00,3.990599e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [47]:
df_train_prep.workclass_goverment.value_counts()

,count
workclass_goverment,
0.0,16886
1.0,3087


**REPRESENTACIÓN "GRÁFICA"**

In [48]:
from sklearn import set_config
set_config(display='diagram')

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('scale', StandardScaler(),
                                 ['age', 'hours_per_week', 'capital']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['workclass', 'marital_status', 'sex'])])

In [49]:
set_config(display='text')

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('scale', StandardScaler(),
                                 ['age', 'hours_per_week', 'capital']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['workclass', 'marital_status', 'sex'])])

In [50]:
clf = GaussianNB()

clf.fit(Xtrain_prep, ytrain)



GaussianNB()

In [51]:
clf.score(Xtrain_prep, ytrain)

0.6626445701697291

In [52]:
clf.score(Xtest_prep, ytest)

0.657388866639968

## CASO UN POCO MAS COMPLEJO: PIPELINE

In [53]:
df2 = pd.read_csv('https://raw.githubusercontent.com/SonikoKatsura/CURSO_PARO_IA/refs/heads/main/Rober_IA/Modulo3/datos/adult2.csv')

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30704 entries, 0 to 30703
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             30704 non-null  int64  
 1   workclass       30704 non-null  object 
 2   marital_status  30704 non-null  object 
 3   sex             30704 non-null  object 
 4   hours_per_week  30704 non-null  int64  
 5   capital         24967 non-null  float64
 6   income          30704 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 1.6+ MB


In [54]:
df2.isnull().sum()

,0
age,0
workclass,0
marital_status,0
sex,0
hours_per_week,0
capital,5737
income,0


In [55]:
target = 'income'

X = df2.drop(columns=target)

y = df2[target]

X.shape, y.shape

((30704, 6), (30704,))

In [56]:
atrib_num = X.select_dtypes(exclude='object').columns.to_list()

atrib_num

['age', 'hours_per_week', 'capital']

In [57]:
atrib_cat = X.select_dtypes(include='object').columns.to_list()

atrib_cat

['workclass', 'marital_status', 'sex']

**El target lo podemos tratar ya:**

In [58]:
label_enc = LabelEncoder()

y_enc = label_enc.fit_transform(y)


In [59]:
y_enc

array([0, 0, 0, ..., 0, 0, 1])

**La novedad es que ahora en las numéricas necesitamos 2 procesados, la imputación de nulos y el escalado. En este caso nos sirve de ayuda "Pipeline" (tratamiento en serie, uno y despues el otro):**

In [60]:
# Transformaciones para las variables numéricas

numeric_transformer = Pipeline(
                        steps=[
                            ('imputer', SimpleImputer(strategy='median')),
                            ('scaler', StandardScaler())
                        ]
                      )

In [61]:
set_config(display='diagram')

numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [62]:
set_config(display='text')

numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

**Para las categóricas no ha cambiado nada. Lo ponemos todo junto:**

In [63]:
preprocessor = ColumnTransformer(
                    transformers=[
                        ('numeric', numeric_transformer, atrib_num),
                        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), atrib_cat)
                    ],
                    remainder='passthrough')


In [64]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

((24563, 6), (6141, 6), (24563,), (6141,))

Respetamos la regla de entrenar con TRAIN:

In [65]:
Xtrain_prep = preprocessor.fit_transform(Xtrain)
Xtest_prep  = preprocessor.transform(Xtest)

Xtrain_prep.shape, Xtest_prep.shape

((24563, 14), (6141, 14))

In [66]:
encoded_cat = preprocessor.named_transformers_['onehot'].get_feature_names_out(atrib_cat)
nombre_columnas = np.concatenate([atrib_num, encoded_cat])

nombre_columnas


array(['age', 'hours_per_week', 'capital', 'workclass_Private',
       'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc',
       'workclass_goverment', 'marital_status_Divorced',
       'marital_status_Married', 'marital_status_Never-married',
       'marital_status_Separated', 'marital_status_Widowed', 'sex_Female',
       'sex_Male'], dtype=object)

In [67]:

df_train_prep = pd.DataFrame(Xtrain_prep, columns=nombre_columnas)

df_train_prep.sample(n=10)


,age,hours_per_week,capital,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_goverment,marital_status_Divorced,marital_status_Married,marital_status_Never-married,marital_status_Separated,marital_status_Widowed,sex_Female,sex_Male
8962,-1.027302,-0.589533,-0.138412,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
22768,-1.256479,-1.197905,-0.138412,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
19895,-1.638439,-2.414648,-0.138412,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
9451,-1.027302,-1.076230,-0.138412,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
9397,0.958894,0.018838,1.797933,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
21437,1.646423,0.018838,-0.902482,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1149,2.410344,1.235582,-0.542485,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
23400,-1.485655,-2.414648,-0.138412,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3401,-0.492557,0.627210,0.122685,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
11902,-0.339773,0.018838,-0.232440,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [68]:
df_train_prep.isnull().sum()

,0
age,0
hours_per_week,0
capital,0
workclass_Private,0
workclass_Self-emp-inc,0
workclass_Self-emp-not-inc,0
workclass_goverment,0
marital_status_Divorced,0
marital_status_Married,0
marital_status_Never-married,0


In [69]:
df_train_prep.describe()

,age,hours_per_week,capital,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_goverment,marital_status_Divorced,marital_status_Married,marital_status_Never-married,marital_status_Separated,marital_status_Widowed,sex_Female,sex_Male
count,2.456300e+04,2.456300e+04,2.456300e+04,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000,24563.000000
mean,1.423226e-16,-8.186443e-17,-5.496198e-16,0.738631,0.036193,0.081668,0.143509,0.138297,0.479502,0.322721,0.031837,0.027643,0.324065,0.675935
std,1.000020e+00,1.000020e+00,1.000020e+00,0.439389,0.186773,0.273863,0.350598,0.345219,0.499590,0.467527,0.175568,0.163952,0.468034,0.468034
min,-1.638439e+00,-2.414648e+00,-3.789130e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-7.981257e-01,1.883839e-02,-5.426165e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.105965e-01,1.883839e-02,-1.384119e-01,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,6.533249e-01,6.272100e-01,2.937920e-01,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,3.938187e+00,1.235582e+00,4.438225e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [70]:
df_train_prep.workclass_goverment.value_counts()

,count
workclass_goverment,
0.0,21038
1.0,3525


In [71]:
from sklearn import set_config
set_config(display='diagram')

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'hours_per_week', 'capital']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['workclass', 'marital_status', 'sex'])])

In [72]:
set_config(display='text')

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'hours_per_week', 'capital']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['workclass', 'marital_status', 'sex'])])